In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Konfigurasi pengurusan hingar dengan Estimator

<Accordion>
<AccordionItem title="Versi pakej">

Kod di halaman ini dibangunkan menggunakan keperluan berikut.
Kami syorkan menggunakan versi ini atau yang lebih baru.

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Terdapat beberapa cara untuk menguruskan hingar, biasanya dengan menggunakan pelbagai teknik pengurangan ralat dan penindasan ralat untuk mengelakkan ralat sebelum ia berlaku. Teknik-teknik ini biasanya menyebabkan overhed pra-pemprosesan. Oleh itu, penting untuk mencapai keseimbangan antara menyempurnakan keputusan kamu dan memastikan kerja kamu selesai dalam masa yang munasabah.

Estimator menyokong teknik pengurusan hingar berikut. Lihat [Teknik pengurangan ralat dan penindasan ralat](error-mitigation-and-suppression-techniques) untuk penjelasan setiap teknik. Lihat bahagian [Tetapan ralat tersuai](#advanced-error) untuk arahan mengaktifkan teknik-teknik ini.

- [dynamical decoupling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options#dynamicaldecouplingoptions)
- [Pauli twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
- [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
- [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec)
- [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
- [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation)
- [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne)

<span id="resilience"></span>
## Tahap daya tahan
`resilience_level` menentukan berapa banyak daya tahan yang perlu dibina terhadap ralat. Tahap yang lebih tinggi menghasilkan keputusan yang lebih tepat, dengan kos masa pemprosesan yang lebih lama. Tahap daya tahan boleh digunakan untuk mengkonfigurasi pertukaran kos/ketepatan apabila menggunakan pengurusan hingar pada pertanyaan primitif kamu. Pengurusan hingar mengurangkan ralat (berat sebelah) dalam keputusan dengan memproses output daripada koleksi, atau ensemble, Circuit yang berkaitan. Tahap pengurangan ralat bergantung pada kaedah yang digunakan. Tahap daya tahan mengabstrak pilihan terperinci kaedah pengurusan hingar untuk membolehkan pengguna menilai pertukaran kos/ketepatan yang sesuai untuk aplikasi mereka.

Memandangkan ini, setiap tahap sepadan dengan kaedah atau kaedah-kaedah dengan tahap overhed pensampelan kuantum yang semakin meningkat untuk membolehkan kamu bereksperimen dengan pertukaran masa-ketepatan yang berbeza. Jadual berikut menunjukkan tahap dan kaedah yang tersedia untuk setiap primitif.

<span id="resilience-table"></span>

| Tahap daya tahan | Penerangan | Teknik |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0 | Tiada pengurangan | Tiada |
| 1 [Lalai] | Kos pengurangan minimum: Kurangkan ralat yang berkaitan dengan ralat readout | Pengukuran twirling [Twirled Readout Error eXtinction (TREX)](/guides/error-mitigation-and-suppression-techniques#trex) |
| 2 | Kos pengurangan sederhana. Biasanya mengurangkan berat sebelah dalam Estimator, tetapi tidak dijamin bebas-berat sebelah. | Tahap 1 + [Zero Noise Extrapolation (ZNE)](/guides/error-mitigation-and-suppression-techniques#zne) dan twirling Gate

> **Info:** Tahap daya tahan kini dalam beta jadi overhed pensampelan dan
> kualiti penyelesaian akan berbeza dari Circuit ke Circuit. Ciri baru,
> pilihan lanjutan, dan alatan pengurusan akan dikeluarkan secara bergilir.
> Kaedah pengurusan hingar khusus tidak dijamin digunakan pada setiap tahap daya tahan.

### Konfigurasi Estimator dengan tahap daya tahan
Kamu boleh menggunakan tahap daya tahan untuk menentukan teknik pengurusan hingar, atau kamu boleh menetapkan teknik tersuai secara individu seperti yang diterangkan dalam [Tetapan ralat tersuai](#advanced-error).

> **Note:** Sebarang pilihan yang kamu tentukan secara manual selain tahap daya tahan diterapkan sebagai tambahan kepada set pilihan asas yang ditentukan oleh tahap daya tahan. Oleh itu, pada dasarnya, kamu boleh menetapkan tahap daya tahan ke 1, tetapi kemudian mematikan pengurangan ukuran, walaupun ini tidak disyorkan.
> 
> Sebagai contoh, menetapkan tahap daya tahan ke 0 mematikan `zne_mitigation`, tetapi `estimator.options.resilience.zne_mitigation = True` mengatasi nilai tersebut.

### Contoh
Kod berikut mengaktifkan ZNE, TREX, dan twirling Gate dengan
menetapkan `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Tetapan pengurusan hingar tersuai
Kamu boleh menghidupkan dan mematikan kaedah pengurusan hingar individu menggunakan [pilihan Estimator](/guides/estimator-options).

> **Note:** Tidak semua pilihan berfungsi bersama pada semua jenis Circuit. Lihat [jadual keserasian ciri](estimator-options#options-compatibility-table) untuk butiran.

### Contoh

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(
    f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}"
)
print(
    f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}"
)

>>> gate twirling is turned on: True
>>> measurement error mitigation is turned on: True


## Turn off all error mitigation

For instructions to turn off all error mitigation see the [Turn off all error suppression and mitigation](estimator-options#no-error-mitigation) section in the Estimator options guide.

## Next steps

<Admonition type="tip" title="Recommendations">
  - Walk through an example that uses error mitigation in the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
  - Learn more about [error mitigation and error suppression techniques](error-mitigation-and-suppression-techniques).
  - Explore [Estimator options](/docs/guides/estimator-options).
  - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
</Admonition>